In [1]:
# ============================
# Cell 1: Install Dependencies
# ============================
!pip install -qU langchain langchain-google-genai chromadb pandas
!pip install -qU langchain-community langchain-text-splitters langchain-classic gradio


In [2]:
# ============================
# Cell 2: Imports & API Key
# ============================
import os
import time
import pandas as pd
from getpass import getpass

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

print("✅ Imports loaded.")

# ---- Gemini API Key Handling for Colab ----
if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    print("🔑 No GOOGLE_API_KEY found in environment.")
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")
else:
    print("🔑 GOOGLE_API_KEY already set in environment.")

print("Setup complete.")


✅ Imports loaded.
🔑 No GOOGLE_API_KEY found in environment.
Enter your Gemini API Key: ··········
Setup complete.


In [3]:
# ===========================================
# Cell 3 (Fixed): Load CSV, Chunk Text, Build/Load Chroma
# ===========================================
import os
import pandas as pd

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 👇 Change this path to your uploaded CSV, e.g. "/content/mtsamples.csv"
DATA_FILE_PATH = "/content/mtsamples.csv"
CHROMA_PATH = "/content/chroma_db_medical"

# Toggle: use only a subset of rows for faster experimentation
USE_SUBSET = True      # set to False when you want the full dataset
MAX_ROWS = 1500        # number of rows to use when USE_SUBSET = True

print(f"📂 Data file: {DATA_FILE_PATH}")
print(f"💾 Chroma path: {CHROMA_PATH}\n")

# If a Chroma DB already exists, just load it and skip recomputing embeddings
if os.path.exists(CHROMA_PATH) and len(os.listdir(CHROMA_PATH)) > 0:
    print("📦 Found existing Chroma DB. Loading it instead of rebuilding embeddings...")
    embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004")
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=CHROMA_PATH
    )
    print("✅ Chroma DB loaded successfully. Skipping embedding step.")
else:
    print("❌ No existing Chroma DB found. Building a new one from CSV...")

    # --- Load CSV with pandas ---
    if USE_SUBSET:
        df = pd.read_csv(DATA_FILE_PATH).head(MAX_ROWS)
        print(f"📄 Loaded {len(df)} rows (subset mode ON, MAX_ROWS={MAX_ROWS}).")
    else:
        df = pd.read_csv(DATA_FILE_PATH)
        print(f"📄 Loaded {len(df)} rows (full dataset).")

    # Build Document list manually (adapt columns to your CSV)
    docs = []
    for idx, row in df.iterrows():
        specialty = row.get("medical_specialty", "")
        description = row.get("description", "")
        transcription = row.get("transcription", "")

        text = (
            f"Row: {idx}\n"
            f"Specialty: {specialty}\n"
            f"Description: {description}\n\n"
            f"Transcription:\n{transcription}"
        )

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "row": int(idx),
                    "source": os.path.basename(DATA_FILE_PATH),
                },
            )
        )

    # --- Split into fewer, larger chunks to reduce embedding calls ---
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,   # larger chunk = fewer calls to embeddings
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    documents = text_splitter.split_documents(docs)

    print(f"🧩 Total chunks created after splitting: {len(documents)}")

    print("\n2. Initializing Gemini Embedding Model (text-embedding-004)...")
    embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004")

    print("3. Creating Chroma vector store and running embeddings (heavy step)...")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=CHROMA_PATH,
    )

    print("✅ Vector Store creation complete! Medical KB is ready for RAG.")


📂 Data file: /content/mtsamples.csv
💾 Chroma path: /content/chroma_db_medical

📦 Found existing Chroma DB. Loading it instead of rebuilding embeddings...


/tmp/ipython-input-2586874963.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


✅ Chroma DB loaded successfully. Skipping embedding step.


In [4]:
# ===========================================
# Cell 4: Build the RAG Retrieval Chain
# ===========================================
print("1. Initializing Gemini LLM...")
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
)

print("2. Setting up the Chroma Retriever...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

system_prompt_template = (
    "You are a helpful and extremely safe medical assistant. Your primary task is to answer the user's question "
    "**ONLY** based on the provided clinical context below. You must not use any external knowledge. "
    "If the context does not contain the answer, you must state clearly: "
    "'I cannot provide an answer based on the provided medical context.' "
    "For every piece of information you provide, you **MUST** include a citation, referencing the original "
    "source document using the format: [Source: Row X]. The context documents contain a 'row' metadata field "
    "that identifies the source.\n\n"
    "Context:\n{context}"
)

print("3. Defining the Citation-Enforcing Prompt...")
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt_template),
        ("human", "{input}"),
    ]
)

print("4. Creating the Document Combination Chain...")
document_chain = create_stuff_documents_chain(llm, prompt)

print("5. Creating the final Retrieval Chain...")
rag_chain = create_retrieval_chain(retriever, document_chain)

print("✅ RAG Pipeline built successfully! Ready for queries.")


1. Initializing Gemini LLM...
2. Setting up the Chroma Retriever...
3. Defining the Citation-Enforcing Prompt...
4. Creating the Document Combination Chain...
5. Creating the final Retrieval Chain...
✅ RAG Pipeline built successfully! Ready for queries.


In [13]:
# ===========================================
# Cell 5: Gradio App (Gen-Z Red & Black Minimalist)
# ===========================================
import gradio as gr

def medical_assistant_response(query: str) -> str:
    if not query.strip():
        return "❗ Please enter a question about the medical transcriptions."

    try:
        # Run RAG chain
        response = rag_chain.invoke({"input": query})

        # 1) Normal assistant answer (whatever Gemini returns, with citations)
        answer = response.get("answer", "").strip()

        # 2) System / evidence view: show retrieved context rows + snippets
        ctx_docs = response.get("context", [])

        if ctx_docs:
            evidence_lines = []
            for i, doc in enumerate(ctx_docs, start=1):
                row_id = doc.metadata.get("row", "Unknown")
                # short snippet so it doesn't explode the UI
                snippet = doc.page_content.replace("\n", " ")
                if len(snippet) > 400:
                    snippet = snippet[:400] + "..."
                evidence_lines.append(
                    f"**Context {i} — Row {row_id}:** {snippet}"
                )
            evidence_block = "\n\n".join(evidence_lines)
        else:
            evidence_block = "_No context documents were returned by the retriever._"

        # Final combined message: answer + system evidence
        final_msg = (
            "### 💬 Assistant Answer\n"
            f"{answer}\n\n"
            "---\n\n"
            "### 🧠 System Evidence (Retrieved Context)\n"
            f"{evidence_block}"
        )
        return final_msg

    except Exception as e:
        return f"⚠️ Error: Something went wrong in the pipeline.\n\nDetails: `{str(e)}`"


# Build the Gradio interface (no CSS parameter - we'll inject it via HTML)
with gr.Blocks() as demo:

    # Inject custom CSS via HTML component
    gr.HTML("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

    * {
        font-family: 'Inter', sans-serif !important;
    }

    body, .gradio-container {
        background: linear-gradient(135deg, #000000 0%, #0a0a0a 100%) !important;
    }

    /* Main container styling */
    .contain {
        background: #1a1a1a !important;
        border-radius: 24px !important;
        padding: 2rem !important;
        box-shadow: 0 8px 32px rgba(255, 0, 0, 0.1) !important;
    }

    /* Input box styling */
    textarea {
        background: #2a2a2a !important;
        border: 2px solid #3a3a3a !important;
        border-radius: 16px !important;
        color: #ffffff !important;
        font-size: 15px !important;
        transition: all 0.3s ease !important;
        padding: 16px !important;
    }

    textarea:focus {
        border-color: #FF0000 !important;
        box-shadow: 0 0 0 3px rgba(255, 0, 0, 0.3) !important;
        outline: none !important;
    }

    textarea::placeholder {
        color: #b0b0b0 !important;
        opacity: 0.6 !important;
    }

    /* Button styling */
    button, .gr-button {
        background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%) !important;
        border: none !important;
        border-radius: 12px !important;
        color: #ffffff !important;
        font-weight: 600 !important;
        font-size: 15px !important;
        padding: 14px 28px !important;
        cursor: pointer !important;
        transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important;
        text-transform: uppercase !important;
        letter-spacing: 0.05em !important;
        box-shadow: 0 4px 16px rgba(255, 0, 0, 0.3) !important;
    }

    button:hover, .gr-button:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 8px 24px rgba(255, 0, 0, 0.4) !important;
    }

    button:active, .gr-button:active {
        transform: translateY(0px) !important;
    }

    /* Output box styling */
    .markdown {
        background: #2a2a2a !important;
        border: 1px solid #3a3a3a !important;
        border-radius: 16px !important;
        padding: 24px !important;
        color: #ffffff !important;
        line-height: 1.7 !important;
        font-size: 15px !important;
    }

    .markdown h3 {
        color: #FF0000 !important;
        margin-top: 1.5rem !important;
        margin-bottom: 1rem !important;
        font-size: 1.2rem !important;
    }

    .markdown hr {
        border-color: #3a3a3a !important;
        margin: 1.5rem 0 !important;
    }

    .markdown strong {
        color: #FF0000 !important;
        font-weight: 600 !important;
    }

    .markdown code {
        background: #3a3a3a !important;
        color: #FF0000 !important;
        padding: 2px 6px !important;
        border-radius: 4px !important;
        font-size: 0.9em !important;
    }

    /* Label styling */
    label, .label {
        color: #ffffff !important;
        font-weight: 600 !important;
        font-size: 14px !important;
        text-transform: uppercase !important;
        letter-spacing: 0.05em !important;
        margin-bottom: 8px !important;
    }

    /* Examples styling */
    .examples {
        background: #2a2a2a !important;
        border-radius: 16px !important;
        padding: 20px !important;
        margin-top: 24px !important;
    }

    .examples button {
        background: #3a3a3a !important;
        border: 1px solid #3a3a3a !important;
        color: #ffffff !important;
        text-transform: none !important;
        font-weight: 400 !important;
        box-shadow: none !important;
        transition: all 0.2s ease !important;
    }

    .examples button:hover {
        background: #2a2a2a !important;
        border-color: #FF0000 !important;
        transform: none !important;
    }

    /* Title card styling */
    .title-card {
        background: linear-gradient(135deg, #1a1a1a 0%, #2a2a2a 100%) !important;
        border: 1px solid #3a3a3a !important;
        border-radius: 20px !important;
        padding: 32px !important;
        margin-bottom: 32px !important;
        text-align: center !important;
        box-shadow: 0 8px 32px rgba(255, 0, 0, 0.15) !important;
    }

    .title-card h1 {
        font-size: 2.5rem !important;
        margin-bottom: 12px !important;
        background: linear-gradient(135deg, #FF0000, #ff4444) !important;
        -webkit-background-clip: text !important;
        -webkit-text-fill-color: transparent !important;
        background-clip: text !important;
    }

    .title-card p {
        color: #b0b0b0 !important;
        font-size: 1rem !important;
        line-height: 1.6 !important;
        margin-bottom: 0 !important;
    }

    /* Disclaimer styling */
    .disclaimer {
        background: #0a0a0a !important;
        border-left: 4px solid #FF0000 !important;
        padding: 16px 20px !important;
        border-radius: 8px !important;
        margin-top: 16px !important;
        font-size: 13px !important;
        color: #b0b0b0 !important;
    }

    /* Scrollbar styling */
    ::-webkit-scrollbar {
        width: 10px !important;
        height: 10px !important;
    }

    ::-webkit-scrollbar-track {
        background: #1a1a1a !important;
        border-radius: 10px !important;
    }

    ::-webkit-scrollbar-thumb {
        background: #FF0000 !important;
        border-radius: 10px !important;
    }

    ::-webkit-scrollbar-thumb:hover {
        background: #CC0000 !important;
    }
    </style>
    """)

    # Title Section
    gr.HTML("""
    <div class="title-card">
        <h1>🩺 MED-RAG</h1>
        <p>AI-powered medical transcript analysis using Gemini + Chroma RAG</p>
        <p style="margin-top: 8px; font-size: 0.9rem;">Ask questions. Get answers. See evidence.</p>
    </div>
    """)

    # Main Input/Output Section
    with gr.Row():
        with gr.Column(scale=4):
            input_box = gr.Textbox(
                label="Your Question",
                placeholder="What were the presenting symptoms in patients with acute appendicitis?",
                lines=3
            )
        with gr.Column(scale=1, min_width=120):
            submit_btn = gr.Button("⚡ Analyze")

    # Output Section
    output_box = gr.Markdown(label="Response")

    # Examples Section
    gr.Examples(
        examples=[
            ["Summarize the key symptoms described for migraine cases in this dataset."],
            ["What postoperative care instructions are commonly given after tonsillectomy?"],
            ["List any medications mentioned for treating hypertension in the transcripts."],
            ["Describe how patients with COPD are typically examined in these notes."],
        ],
        inputs=input_box,
        label="💡 Try these questions"
    )

    # Disclaimer
    gr.HTML("""
    <div class="disclaimer">
        <strong>⚠️ Disclaimer:</strong> This is an academic project.
        Not intended for real clinical use. Always consult qualified healthcare professionals for medical advice.
    </div>
    """)

    # Event handlers
    submit_btn.click(
        fn=medical_assistant_response,
        inputs=input_box,
        outputs=output_box
    )

    input_box.submit(
        fn=medical_assistant_response,
        inputs=input_box,
        outputs=output_box
    )

print("🚀 Ready to launch Gradio. Now run: demo.launch(share=True)")

🚀 Ready to launch Gradio. Now run: demo.launch(share=True)


In [14]:
# ===========================================
# Cell 6: Launch Gradio
# ===========================================
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a34d5ebdcf36162942.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ===========================================
# Cell 7: Batch Evaluation with Updated Queries
# ===========================================
import pandas as pd
import time

queries = [
    # Procedure-specific
    "Which technique was used for laparoscopic cholecystectomy in the dataset?",
    "Describe the incision type documented for carpal tunnel release.",
    "What type of anesthesia is most commonly mentioned for colonoscopy procedures?",
    "Summarize the MRI lumbar spine findings in any available transcription.",
    "How are patients positioned for total knee arthroplasty according to the notes?",
    "Which sutures are used to close fascia in hernia repair operations?",
    "Summarize the typical intraoperative findings during cystoscopy.",
    "What complications, if any, are reported during cataract surgeries?",
    "What is the estimated blood loss reported for lumbar fusion cases?",
    "Which specific medication is injected during epidural steroid injections?",

    # Symptoms / history
    "What presenting symptoms are associated with allergic rhinitis in the dataset?",
    "Describe the documented history of COPD patients in these transcriptions.",
    "Which symptoms are reported for acute appendicitis cases?",
    "What vital signs are recorded for patients presenting with chest pain?",
    "What physical exam findings appear in otitis media descriptions?",
    "What previous surgeries are mentioned for patients with hip fractures?",
    "Which allergies are listed for patients with skin abscesses?",
    "Is any family history of cardiovascular disease reported in the notes?",
    "What neurological symptoms are mentioned for patients with headaches?",
    "Summarize the social history for any patient evaluated for depression.",

    # Management / plan
    "What postoperative plan is described for tonsillectomy patients?",
    "Which medications are prescribed for hypertension within the dataset?",
    "What type of physical therapy is recommended after rotator cuff repair?",
    "What diet recommendations are given to patients with diabetes mellitus?",
    "What follow-up or wound care instructions are documented?",

    # Out-of-scope sanity checks
    "Who invented the stethoscope?",
    "What is the capital city of France?",
    "What is the current stock price of Pfizer?",
    "Summarize the storyline of the movie 'The Doctor'.",
    "How do I bake a basic vanilla cake?",

    # High-level dataset queries
    "Compare the anesthesia types used across orthopedic surgeries in the dataset.",
    "List common cardiovascular risk factors mentioned across different patients.",
    "What appear to be the most frequent postoperative diagnoses in these transcriptions?"
]

results = []

print(f"Starting evaluation of {len(queries)} queries...\n")

for i, query in enumerate(queries):
    print(f"Running query {i+1}/{len(queries)}: {query}")
    try:
        response = rag_chain.invoke({"input": query})

        source_docs_combined = []
        if response.get("context"):
            for doc in response["context"]:
                row_id = doc.metadata.get("row", "Unknown")
                content = doc.page_content.replace("\n", " ").strip()
                source_docs_combined.append(f"[SOURCE ROW {row_id}]: {content}")

        final_source_string = " ---|||--- ".join(source_docs_combined)

        results.append({
            "Query": query,
            "Answer": response.get("answer", ""),
            "Source_Documents": final_source_string
        })
    except Exception as e:
        print(f"❌ Error on query: {query}\n{e}")
        results.append({
            "Query": query,
            "Answer": "ERROR",
            "Source_Documents": "Error during retrieval."
        })

    time.sleep(5)  # preserve original throttling logic

df_results = pd.DataFrame(results)
output_path = "/content/rag_evaluation_results_with_content.csv"
df_results.to_csv(output_path, index=False)

print("\n✅ Evaluation complete!")
print(f"Results saved to: {output_path}")


Starting evaluation of 33 queries...

Running query 1/33: Which technique was used for laparoscopic cholecystectomy in the dataset?
Running query 2/33: Describe the incision type documented for carpal tunnel release.
Running query 3/33: What type of anesthesia is most commonly mentioned for colonoscopy procedures?
Running query 4/33: Summarize the MRI lumbar spine findings in any available transcription.
Running query 5/33: How are patients positioned for total knee arthroplasty according to the notes?
Running query 6/33: Which sutures are used to close fascia in hernia repair operations?
Running query 7/33: Summarize the typical intraoperative findings during cystoscopy.
Running query 8/33: What complications, if any, are reported during cataract surgeries?
Running query 9/33: What is the estimated blood loss reported for lumbar fusion cases?
Running query 10/33: Which specific medication is injected during epidural steroid injections?
Running query 11/33: What presenting symptoms are